# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# One row:
#   One row represents a single unique content_id (i.e. page on a website) on a specific date
#   within a month with it's search and engagement metrics calculated for a day.


# Grain:
#   One row per (content_hash_id × report_date)


# Tables Used:
#   Primary table -> fact_content_daily_performance (partitioned by month)
#   Secondary tables (context only) -> dim_content, dim_clients


# Time Window:
#   Development window -> month = '2026-03'
#   Rolling window per row -> Each row shows that specific day's metrics.


# Predict/Rank (label/proxy):
#   This is unsupervised learning. There is no pre-existing label.
#   We have 5 numerical features (impressions, clicks, CTR, position, engagement).
#   K-Means will assign each content_id to one of 4 clusters.
#   Cluster membership is the outcome, created AFTER training.


# Data to deliberately exclude:
#   Query text (anonymized hashes) -> salted pseudonyms that can't be reversed to recover text
#   Rows with zero impressions -> Can't characterize page performance
#   Rows with missing clicks data -> Incomplete activity data
#   Brand and navigational queries (if identifiable) -> Behave differently from informational queries
#   Days with insufficient data -> Very new content may cluster artificially

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# Features:
#   1. impressions_90d — total impressions in 90-day window
#   2. clicks_90d — total clicks in 90-day window
#   3. avg_position_90d — average search position
#   4. content_total_impressions_90d — total impressions for this content
#   5. content_visible_query_count — how many different queries drive traffic


# Context:
#   1. client_hash_id — which client owns this content
#   2. content_hash_id — which content piece
#   3. query_hash_id — which query drove this row
#   4. window_start — when the 90-day window begins
#   5. window_end — when the 90-day window ends


# Label:
#   None - unsupervised clustering


# Excluded:
#   1. query_hash_id
#      Why: Anonymized as hash, can't recover query text for semantic clustering

#   2. query_char_count
#      Why: Redundant with query_hash_id. Adds noise & doesn't reflect content quality

#   3. query_token_count
#      Why: Same as above; query-level detail, not content quality

#   4. impressions_last30 & clicks_last30
#      Why: Window overlap. The last 30 days of the 90-day window overlap with the label window.

#   5. impressions_prev30 & clicks_prev30
#      Why: These are pre-window performance. Conflates old vs. new data.
#           Use aggregated 90-day metrics instead for consistency.

#   6. avg_position_last30 & avg_position_prev30
#      Why: Same reasoning as impressions/clicks windows.
#           Use avg_position_90d for consistency.

#   7. rare_query_count
#      Why: Encodes search tail behavior specific to this query. Not a general
#           content quality signal.

#   8. rare_impressions_share
#      Why: Tail-specific metric, doesn't generalize across queries.

#   9. anonymized_impressions_share
#      Why: Aggregates anonymized data (tail impressions). Too opaque to interpret for clustering.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

print("Connected to HuggingFace warehouse via DuckDB")

Connected to HuggingFace warehouse via DuckDB


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.